# Customer Support Agent — Evaluation, LangSmith Observability & OpenTelemetry

**Goal:** Build a small e-commerce customer support agent with **LangGraph**, trace every run in **LangSmith**, and learn how to evaluate it like a real ML system.

By the end of this notebook you will have:
1. A LangGraph agent that classifies support tickets into clear categories
2. A synthetic ticket dataset with ground-truth labels
3. LangSmith traces for every prediction (clickable in the UI), emitted through OpenTelemetry
4. Accuracy / precision / recall metrics
5. A CSV of results you can open in a spreadsheet and annotate
6. A simple loop: **inspect failures → tweak the prompt → re-run → compare**

---

## The workflow you'll follow

1. **Run the agent** on the synthetic tickets.
2. **Look at the metrics** and open the exported CSV in Google Sheets / Excel.
3. **Add validator comments** in the `validator_comment` column — flag anything that looks wrong, ambiguous, or surprising.
4. **Cluster the failures** into 2–4 *failure categories* (e.g., "misroutes complaints as questions", "confuses refund vs. return").
5. **Pick the one failure category that matters most** for your use case.
6. **Tweak the classifier prompt** at the bottom of the notebook to address it.
7. **Re-run** the agent and compare the new metrics + LangSmith/OpenTelemetry traces.

## 1. Install dependencies

In [1]:
%pip install --quiet langgraph "langsmith[otel]>=0.4.25" langchain-openai langchain-core pandas scikit-learn pydantic tqdm opentelemetry-sdk opentelemetry-exporter-otlp


Note: you may need to restart the kernel to use updated packages.


## 2. Set up API keys, LangSmith tracing, and OpenTelemetry

You need two keys:
- **`OPENAI_API_KEY`** — get one at [platform.openai.com/api-keys](https://platform.openai.com/api-keys)
- **`LANGSMITH_API_KEY`** — get one at [smith.langchain.com](https://smith.langchain.com) → Settings → API Keys

Once these are set, every LLM call is traced in LangSmith. We also enable OpenTelemetry so each ticket-level eval run can carry standard span metadata like ticket id, prompt version, expected label, predicted label, and correctness.

In [2]:
import os
import getpass
from pathlib import Path

# Clear stale tracing variables so an old endpoint does not make OTel export to a 404 URL.
for k in [
    "LANGCHAIN_API_KEY",
    "LANGCHAIN_ENDPOINT",
    "LANGCHAIN_TRACING_V2",
    "OTEL_EXPORTER_OTLP_ENDPOINT",
    "OTEL_EXPORTER_OTLP_TRACES_ENDPOINT",
    "OTEL_EXPORTER_OTLP_HEADERS",
    "OTEL_EXPORTER_OTLP_TRACES_HEADERS",
    "OTEL_EXPORTER_OTLP_PROTOCOL",
]:
    os.environ.pop(k, None)

def _load_dotenv(path: str) -> dict:
    """Parse a simple KEY=VALUE .env file. Returns {} if the file doesn't exist."""
    env_path = Path(path)
    if not env_path.exists():
        return {}
    values = {}
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip()
    return values

_dotenv = _load_dotenv(".env.demo")

def _set(key: str, prompt: str):
    # .env.demo always wins over a stale value left in os.environ by an earlier
    # kernel run, so rotating the key in .env.demo takes effect without a kernel restart.
    if _dotenv.get(key):
        os.environ[key] = _dotenv[key]
    elif not os.environ.get(key):
        os.environ[key] = getpass.getpass(prompt)

_set("OPENAI_API_KEY",    "OpenAI API key: ")
_set("LANGSMITH_API_KEY", "LangSmith API key: ")

# LangSmith is still the evaluation/debugging UI.
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "customer-support-evals"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"  # use https://eu.api.smith.langchain.com for EU accounts

# OpenTelemetry is the standard tracing layer. LangSmith receives the emitted spans.
os.environ["LANGSMITH_OTEL_ENABLED"] = "true"
os.environ["OTEL_SERVICE_NAME"] = "customer-support-evals-notebook"
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://api.smith.langchain.com/otel"
os.environ["OTEL_EXPORTER_OTLP_TRACES_ENDPOINT"] = "https://api.smith.langchain.com/otel/v1/traces"
os.environ["OTEL_EXPORTER_OTLP_PROTOCOL"] = "http/protobuf"

otel_headers = f"x-api-key={os.environ['LANGSMITH_API_KEY']},Langsmith-Project={os.environ['LANGSMITH_PROJECT']}"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = otel_headers
os.environ["OTEL_EXPORTER_OTLP_TRACES_HEADERS"] = otel_headers

print("Traces will appear in LangSmith project:", os.environ["LANGSMITH_PROJECT"])
print("OpenTelemetry service name:", os.environ["OTEL_SERVICE_NAME"])
print("OpenTelemetry traces endpoint:", os.environ["OTEL_EXPORTER_OTLP_TRACES_ENDPOINT"])


Traces will appear in LangSmith project: customer-support-evals
OpenTelemetry service name: customer-support-evals-notebook
OpenTelemetry traces endpoint: https://api.smith.langchain.com/otel/v1/traces


## 3. Define the ticket categories

We keep the label space small and unambiguous on purpose — when there are 50 categories, *everything* looks like a model failure. Five is a good teaching size.

| Category | What belongs here |
|---|---|
| `order_status` | "Where is my order?", tracking, delivery ETA |
| `refund_request` | Customer wants money back, return-for-refund |
| `product_issue` | Item arrived broken, wrong, defective, or not as described |
| `account_help` | Login, password, address, payment method changes |
| `other` | Anything that doesn't fit above (general questions, feedback) |

In [3]:
CATEGORIES = [
    "order_status",
    "refund_request",
    "product_issue",
    "account_help",
    "other",
]

## 4. Load the golden ticket dataset

This dataset is pulled directly from the course evaluation worksheet
(https://docs.google.com/spreadsheets/d/1MVIxuy2C6FNDcDVEgmZjV_LqX9Fd4rCRbpv2qRabeSY) —
150 tickets, each with a human-assigned `true_category`. Using this fixed, pre-labeled
set (instead of generating fresh synthetic paraphrases each run) means the `true_category`
column here matches exactly what Step 1-3 of that worksheet used to annotate and cluster
classifier failures, so prompt changes can be checked against the same ground truth.


In [4]:
# Golden ticket dataset — pulled directly from the course evaluation worksheet
# (https://docs.google.com/spreadsheets/d/1MVIxuy2C6FNDcDVEgmZjV_LqX9Fd4rCRbpv2qRabeSY),
# so every row's true_category matches what Step 1-3 of that worksheet used to grade failures.
GOLDEN_TICKETS = [
    ("Hi, I ordered a blender 5 days ago and the tracking page hasn't updated. Can you tell me where it is?", "order_status"),
    ("Hey, placed an order for a blender 5 days back and the tracking still shows nothing. Where is it?", "order_status"),
    ("So I ordered a blender on Monday, supposedly it shipped, but the tracking page is dead. What's going on?", "order_status"),
    ("My package was supposed to arrive yesterday but it's still not here. Order #44812.", "order_status"),
    ("Order #44812 was due yesterday and hasn't shown up. Any update?", "order_status"),
    ("Where's my package? Was supposed to arrive yesterday — order #44812 if that helps.", "order_status"),
    ("Tracking link in the email just spins forever. Order #88421.", "order_status"),
    ("Hey, the tracking link you emailed me for order 88421 doesn't load — just keeps spinning. Help?", "order_status"),
    ("Tracking URL is broken. Order 88421. Can you tell me where the package is?", "order_status"),
    ("Order shows delivered last Tuesday but it's not at my door, my neighbor's, or the mailroom. What now?", "order_status"),
    ("My order is marked delivered as of last Tuesday but it's nowhere to be found. Help.", "order_status"),
    ("Says delivered Tuesday — checked the porch, the neighbors, the mailroom. Nothing. What do I do?", "order_status"),
    ("I never received my order and I want my money back.", "refund_request"),
    ("I haven't received my order yet — please refund me.", "refund_request"),
    ("Still no package. I want a refund.", "refund_request"),
    ("Order shows delivered but I never got it. Refund please.", "refund_request"),
    ("Package says delivered but I don't have it. I'd like a refund.", "refund_request"),
    ("Marked as delivered, I never got it. Just refund me at this point.", "refund_request"),
    ("The website said 2-day shipping. It's been 9 days. Are you kidding me?", "order_status"),
    ("Promised 2-day delivery. Day nine. This is a joke.", "order_status"),
    ("You advertised 2-day shipping. Where is my order? It's been more than a week.", "order_status"),
    ("My item has been stuck in customs for 3 weeks. What can you do?", "order_status"),
    ("Customs has been holding my order for three weeks. Any way to speed this up?", "order_status"),
    ("It's been 21 days in customs. I need an update on when this will arrive.", "order_status"),
    ("The carrier marked it delivered but it shows the wrong delivery address.", "order_status"),
    ("Carrier delivered to the wrong address. The address on the label is not mine.", "order_status"),
    ("Order was sent to an address I don't recognize and now it's gone. What happened?", "order_status"),
    ("Where exactly is order #50221? I need it for my wedding tomorrow.", "order_status"),
    ("Order #50221 is for my wedding tomorrow. Where is it??", "order_status"),
    ("Wedding is tomorrow and order 50221 still hasn't arrived. Please help.", "order_status"),
    ("I'd like to return the headphones I bought last week and get my money back. They're unopened.", "refund_request"),
    ("Need to return these unopened headphones — refund please.", "refund_request"),
    ("Hey, the headphones I bought last week are still sealed — can I return them for my money back?", "refund_request"),
    ("Please cancel order 99021 and refund my card. I no longer need it.", "refund_request"),
    ("Cancel order 99021 and refund the charge — I changed my mind.", "refund_request"),
    ("Please void order 99021 and put the money back on my card.", "refund_request"),
    ("Where is my refund? I returned the item two weeks ago and still nothing on my card.", "refund_request"),
    ("Returned the item two weeks ago, no refund yet — where is it?", "refund_request"),
    ("I sent the product back 14 days ago. My card still hasn't been credited. What's the holdup?", "refund_request"),
    ("I was charged $89 but the website showed $79 at checkout. Please refund the difference.", "refund_request"),
    ("Charged $89, but the site listed $79. Please refund the $10 difference.", "refund_request"),
    ("You overcharged me by $10. Please credit the difference back to my card.", "refund_request"),
    ("Returning these for store credit is fine but honestly I'd prefer cash back to my original card.", "refund_request"),
    ("Returning these — and please refund to my card, not store credit.", "refund_request"),
    ("Happy to return these but I'd rather get a real refund than store credit.", "refund_request"),
    ("I was double-charged on the same order. Please refund the duplicate transaction.", "refund_request"),
    ("There are two charges on my card for the same order. Please refund one.", "refund_request"),
    ("Card was hit twice for one order — please reverse the duplicate.", "refund_request"),
    ("The price dropped two days after I bought it. Can I get the difference back?", "refund_request"),
    ("Saw the same item dropped in price right after I bought it. Any way to get the difference?", "refund_request"),
    ("Item went on sale right after I ordered. Do you do price adjustments?", "refund_request"),
    ("Cancel my subscription and refund this month's charge — I never used it.", "refund_request"),
    ("Please cancel my subscription and refund this month — I haven't used the service.", "refund_request"),
    ("Subscription cancel + refund of this month, please. I haven't used it once.", "refund_request"),
    ("Your return window says 30 days. I missed it by 2 days. Refund anyway?", "refund_request"),
    ("I'm just past the 30-day return window — would you still take it back for a refund?", "refund_request"),
    ("Return window technically ended 2 days ago. Any chance of a refund?", "refund_request"),
    ("Disputing this charge with my bank — but I'd rather just get a refund directly from you.", "refund_request"),
    ("Was about to dispute with my bank — would rather just get refunded by you instead.", "refund_request"),
    ("Don't want to do a chargeback — please just refund me directly.", "refund_request"),
    ("The coffee maker arrived with a cracked carafe. Really disappointed.", "product_issue"),
    ("Coffee maker showed up with the carafe cracked. Not happy.", "product_issue"),
    ("The carafe on the coffee maker was already cracked when I opened the box.", "product_issue"),
    ("You sent me a size medium shirt but I ordered a large. Second time this has happened.", "product_issue"),
    ("Wrong size again — got a medium, ordered a large. This is twice now.", "product_issue"),
    ("I ordered a large shirt and you sent a medium. Second time this has happened.", "product_issue"),
    ("Item itself works fine but the box was crushed and the instruction manual is missing.", "product_issue"),
    ("Product works, but the box arrived crushed and there's no manual inside.", "product_issue"),
    ("Box was destroyed and the manual is missing. The actual product is fine though.", "product_issue"),
    ("It's defective — third one in a row from your store.", "product_issue"),
    ("Third defective unit from you — this one doesn't work either.", "product_issue"),
    ("Yet another defective product. Three for three from your store.", "product_issue"),
    ("My laptop arrived damaged and I want a full refund, not a replacement.", "refund_request"),
    ("Laptop showed up damaged. I want my money back, not a replacement.", "refund_request"),
    ("Damaged laptop on arrival — please refund me in full, no replacement.", "refund_request"),
    ("Sent the wrong color — I want my money back.", "refund_request"),
    ("Wrong color shipped. Refund please.", "refund_request"),
    ("You sent the wrong color. Just refund me.", "refund_request"),
    ("The product description said 'wireless' but I had to buy a separate dongle to use it. Misleading.", "product_issue"),
    ("Description claimed wireless. I had to buy a dongle separately. Felt misled.", "product_issue"),
    ("Item is technically wireless but only with a separate dongle, which the listing didn't mention.", "product_issue"),
    ("This pillow is way smaller than what the photos suggested. False advertising.", "product_issue"),
    ("Pillow is much smaller than the photos. Misleading.", "product_issue"),
    ("Got the pillow, it's tiny compared to the photos on the site.", "product_issue"),
    ("The blender stopped working after 3 days. Can I get a replacement?", "product_issue"),
    ("Blender died after three days of use. Replacement, please.", "product_issue"),
    ("My blender stopped working three days in — can you send a replacement?", "product_issue"),
    ("Item smells strongly of chemicals — definitely not okay to use around food.", "product_issue"),
    ("Strong chemical smell on the item. I wouldn't use this around food.", "product_issue"),
    ("Whatever you sent reeks of chemicals. Not food-safe.", "product_issue"),
    ("I can't log into my account — it keeps saying my password is wrong even after I reset it.", "account_help"),
    ("Password reset didn't help — still can't log in.", "account_help"),
    ("Login keeps saying wrong password even though I just reset it. Help.", "account_help"),
    ("How do I update the credit card on file? I don't see the option anywhere in settings.", "account_help"),
    ("I need to change my saved card. Can't find the setting.", "account_help"),
    ("Where do I update my credit card info? Settings don't show it.", "account_help"),
    ("Please remove my old shipping address. I moved last month and don't want stuff going there.", "account_help"),
    ("Drop my old shipping address from my account. Just moved.", "account_help"),
    ("Could you delete my previous shipping address? I don't want things sent there.", "account_help"),
    ("I keep getting 2FA codes I didn't request. Is someone trying to access my account?", "account_help"),
    ("2FA codes are showing up that I didn't ask for. Should I be worried?", "account_help"),
    ("Random 2FA codes are arriving by SMS. I didn't request any.", "account_help"),
    ("The website won't let me check out — it keeps logging me out mid-payment.", "account_help"),
    ("Site logs me out every time I try to pay.", "account_help"),
    ("Can't finish my purchase — keeps signing me out at the payment step.", "account_help"),
    ("Your app keeps crashing whenever I try to view my order history.", "account_help"),
    ("App crashes every time I open my order history.", "account_help"),
    ("Whenever I tap on my orders the app force-closes.", "account_help"),
    ("I forgot which email I used to sign up. How do I find my old account?", "account_help"),
    ("Don't remember which email my account is under. How do I look it up?", "account_help"),
    ("Lost track of the email I signed up with — can you help me find the account?", "account_help"),
    ("Need to change my account email to a different address — old one is dead.", "account_help"),
    ("Old email is gone — please switch my account to my new email.", "account_help"),
    ("My signup email is dead. I want to switch the account to a new one.", "account_help"),
    ("The discount code from your newsletter won't apply at checkout. Says 'invalid'.", "account_help"),
    ("Newsletter promo code is being rejected at checkout as invalid.", "account_help"),
    ("My promo code says 'invalid' even though I just got the newsletter.", "account_help"),
    ("Receipt email says I bought it, but the order isn't showing in my account.", "account_help"),
    ("Got a receipt by email but the order isn't in my account's order history.", "account_help"),
    ("Email says my purchase went through but I don't see the order in my account.", "account_help"),
    ("Do you guys ship to Canada? Couldn't find it on the FAQ page.", "other"),
    ("Quick question — do you ship to Canada?", "other"),
    ("Couldn't find shipping info on the site. Do you ship to Canada?", "other"),
    ("Just wanted to say the customer service rep I spoke to yesterday was amazing. Thank you!", "other"),
    ("Big thanks to your customer service team — yesterday's rep was great!", "other"),
    ("The rep I worked with yesterday was wonderful — please pass along my thanks.", "other"),
    ("Is the red version of SKU-1140 back in stock?", "other"),
    ("Restock alert? When will SKU-1140 in red be back?", "other"),
    ("Looking for the red SKU-1140 — is it available again yet?", "other"),
    ("Do you offer a student discount? Couldn't find one at checkout.", "other"),
    ("Any student-discount option? Didn't see one in checkout.", "other"),
    ("Is there a student discount on your store?", "other"),
    ("When's your next sale? My birthday is coming up and I'd love to splurge.", "other"),
    ("Got any sales coming up? Birthday is around the corner.", "other"),
    ("When does your next sale start? Want to time a big purchase.", "other"),
    ("Do you have a sustainability report or info on where your products are sourced?", "other"),
    ("Where do you source your products from? Curious about sustainability.", "other"),
    ("Any sustainability info available for your products?", "other"),
    ("Can I bulk-order 50 units? Looking for B2B pricing.", "other"),
    ("Need 50 units — do you offer B2B pricing?", "other"),
    ("Bulk order question — 50 units, any volume discount?", "other"),
    ("Why is shipping so expensive? Just feedback — not asking about a specific order.", "other"),
    ("Your shipping prices are wild. Just sharing feedback.", "other"),
    ("Shipping costs feel really high. Just venting, not about a specific order.", "other"),
    ("Are the AirPods you sell authentic Apple products?", "other"),
    ("Just want to make sure — the AirPods on your store are genuine Apple, right?", "other"),
    ("Are these AirPods real Apple gear or knock-offs?", "other"),
    ("Do you sell gift cards? Couldn't find them anywhere on the site.", "other"),
    ("Any gift cards available? Don't see them in the menu.", "other"),
    ("Do you offer gift cards?", "other"),
]

tickets = [
    {"id": f"t{i:03d}", "text": text, "true_category": label, "source": "golden"}
    for i, (text, label) in enumerate(GOLDEN_TICKETS)
]

print(f"Loaded {len(tickets)} tickets across {len(CATEGORIES)} categories.")
from collections import Counter
for cat, n in Counter(t["true_category"] for t in tickets).items():
    print(f"  {cat:>15}: {n}")
print()
for t in tickets[:5]:
    print(f"  [{t['true_category']:>15}] {t['text'][:90]}")


Loaded 150 tickets across 5 categories.
     order_status: 24
   refund_request: 42
    product_issue: 24
     account_help: 30
            other: 30

  [   order_status] Hi, I ordered a blender 5 days ago and the tracking page hasn't updated. Can you tell me w
  [   order_status] Hey, placed an order for a blender 5 days back and the tracking still shows nothing. Where
  [   order_status] So I ordered a blender on Monday, supposedly it shipped, but the tracking page is dead. Wh
  [   order_status] My package was supposed to arrive yesterday but it's still not here. Order #44812.
  [   order_status] Order #44812 was due yesterday and hasn't shown up. Any update?


## 5a. What OpenTelemetry adds here

LangSmith is the place where we inspect AI traces and compare baseline vs improved runs. OpenTelemetry is the standard way we emit structured traces from code.

In this notebook, we add one OpenTelemetry parent span around each ticket classification. The LangGraph / LangChain internals still create child spans for the model call, and the parent span carries eval metadata:

- `eval.example_id`: ticket id
- `eval.run_name`: baseline or improved
- `eval.prompt_version`: v1 or v2
- `eval.true_category`: ground-truth label
- `eval.predicted_category`: model output
- `eval.correct`: whether the prediction matched the label
- `eval.reasoning`: model explanation

This makes each row in the CSV traceable back to a specific LangSmith/OpenTelemetry run.

If you see `Failed to export span batch code: 404`, the classifier is still running, but the OTLP exporter is pointed at the wrong URL. This notebook clears stale `OTEL_EXPORTER_*` values and explicitly sends trace spans to `https://api.smith.langchain.com/otel/v1/traces`.


## 5. Build the LangGraph classification agent

LangGraph models an agent as a **graph of nodes**. For a classifier, the graph is tiny — one node that calls the LLM with a structured output schema. We're using LangGraph here (instead of just calling the LLM directly) so the pattern scales to multi-step agents later (e.g., add a retrieval node, a tool-calling node, a confidence-check node).

Because LangSmith tracing is on, **every graph invocation becomes a clickable trace** showing each node's input/output.

In [5]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate

# --- THIS PROMPT IS WHAT YOU'LL TWEAK LATER ---
CLASSIFIER_PROMPT = """You are a triage system for an e-commerce support inbox.

Classify the customer's ticket into EXACTLY ONE of these categories:

- order_status: questions about where an order is, tracking, delivery ETA
- refund_request: the customer wants their money back
- product_issue: the item arrived broken, wrong, defective, or not as described
- account_help: login, password, address, payment method changes
- other: anything that doesn't fit the above (general questions, feedback, browsing)

Return only the category key.

Ticket:
{ticket_text}
"""

class Classification(BaseModel):
    category: Literal["order_status", "refund_request", "product_issue", "account_help", "other"]
    reasoning: str = Field(description="One short sentence explaining the choice.")

class AgentState(TypedDict):
    ticket_text: str
    category: str
    reasoning: str

def build_agent(prompt_template: str):
    """Compile a LangGraph agent. Re-call this any time you change the prompt."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(Classification)
    prompt = ChatPromptTemplate.from_template(prompt_template)

    def classify_node(state: AgentState) -> AgentState:
        result = (prompt | llm).invoke({"ticket_text": state["ticket_text"]})
        return {"ticket_text": state["ticket_text"], "category": result.category, "reasoning": result.reasoning}

    graph = StateGraph(AgentState)
    graph.add_node("classify", classify_node)
    graph.add_edge(START, "classify")
    graph.add_edge("classify", END)
    return graph.compile()

agent = build_agent(CLASSIFIER_PROMPT)

# Smoke test on one ticket.
sample = agent.invoke({"ticket_text": tickets[0]["text"], "category": "", "reasoning": ""})
print("Ticket:    ", tickets[0]["text"])
print("Predicted: ", sample["category"])
print("Reasoning: ", sample["reasoning"])

/Users/arsegas2/Projects/Lesson4/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


/Users/arsegas2/Projects/Lesson4/.venv/lib/python3.9/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Ticket:     Hi, I ordered a blender 5 days ago and the tracking page hasn't updated. Can you tell me where it is?
Predicted:  order_status
Reasoning:  The customer is inquiring about the status and tracking of their order.


/Users/arsegas2/Projects/Lesson4/.venv/lib/python3.9/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Classification(category='...acking of their order.'), input_type=Classification])
  return self.__pydantic_serializer__.to_python(


In [6]:
from opentelemetry import trace

# This tracer creates ticket-level spans. LangSmith receives them because
# LANGSMITH_OTEL_ENABLED=true is set above.
tracer = trace.get_tracer("customer-support-evals")


## 6. Run the agent on every ticket

Each invocation is automatically traced in LangSmith. After this cell finishes, go to your [LangSmith dashboard](https://smith.langchain.com) → project **`customer-support-evals`** → and you'll see every prediction with full input/output/latency.

In [7]:
import pandas as pd
from tqdm import tqdm

def run_predictions(agent, tickets, run_name="baseline", prompt_version="v1") -> pd.DataFrame:
    rows = []
    for t in tqdm(tickets, desc=f"Classifying {run_name}"):
        # One parent span per ticket makes the spreadsheet row traceable in LangSmith.
        with tracer.start_as_current_span("customer_support.classify_ticket") as span:
            span.set_attribute("langsmith.span.kind", "chain")
            span.set_attribute("eval.run_name", run_name)
            span.set_attribute("eval.prompt_version", prompt_version)
            span.set_attribute("eval.example_id", t["id"])
            span.set_attribute("eval.true_category", t["true_category"])
            span.set_attribute("input.ticket_text", t["text"])

            out = agent.invoke({"ticket_text": t["text"], "category": "", "reasoning": ""})
            correct = t["true_category"] == out["category"]

            span.set_attribute("eval.predicted_category", out["category"])
            span.set_attribute("eval.correct", correct)
            span.set_attribute("eval.reasoning", out["reasoning"])
            span.set_attribute("output.category", out["category"])

            rows.append({
                "id": t["id"],
                "ticket_text": t["text"],
                "true_category": t["true_category"],
                "predicted_category": out["category"],
                "reasoning": out["reasoning"],
                "correct": correct,
                "otel_run_name": run_name,
                "otel_prompt_version": prompt_version,
            })
    return pd.DataFrame(rows)

results_v1 = run_predictions(agent, tickets, run_name="baseline", prompt_version="v1")
results_v1.head(10)


Classifying baseline:   0%|          | 0/150 [00:00<?, ?it/s]

/Users/arsegas2/Projects/Lesson4/.venv/lib/python3.9/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Classification(category='...acking of their order.'), input_type=Classification])
  return self.__pydantic_serializer__.to_python(
Classifying baseline:   1%|          | 1/150 [00:00<01:34,  1.57it/s]

Classifying baseline:   1%|▏         | 2/150 [00:01<02:04,  1.19it/s]

/Users/arsegas2/Projects/Lesson4/.venv/lib/python3.9/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Classification(category='... tracking information.'), input_type=Classification])
  return self.__pydantic_serializer__.to_python(
Classifying baseline:   2%|▏         | 3/150 [00:02<01:52,  1.31it/s]

/Users/arsegas2/Projects/Lesson4/.venv/lib/python3.9/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Classification(category='...rder and its delivery.'), input_type=Classification])
  return self.__pydantic_serializer__.to_python(
Classifying baseline:   3%|▎         | 4/150 [00:03<01:56,  1.25it/s]

...

Classifying baseline:  99%|█████████▊| 148/150 [01:44<00:01,  1.38it/s]

Classifying baseline:  99%|█████████▉| 149/150 [01:45<00:00,  1.39it/s]

Classifying baseline: 100%|██████████| 150/150 [01:45<00:00,  1.40it/s]

Classifying baseline: 100%|██████████| 150/150 [01:45<00:00,  1.42it/s]

,id,ticket_text,true_category,predicted_category,reasoning,correct,otel_run_name,otel_prompt_version
0,t000,"Hi, I ordered a blender 5 days ago and the tra...",order_status,order_status,The customer is inquiring about the status and...,True,baseline,v1
1,t001,"Hey, placed an order for a blender 5 days back...",order_status,order_status,The customer is inquiring about the status and...,True,baseline,v1
2,t002,"So I ordered a blender on Monday, supposedly i...",order_status,order_status,The customer is inquiring about the status of ...,True,baseline,v1
3,t003,My package was supposed to arrive yesterday bu...,order_status,order_status,The customer is inquiring about the status of ...,True,baseline,v1
4,t004,Order #44812 was due yesterday and hasn't show...,order_status,order_status,The customer is inquiring about the status of ...,True,baseline,v1
5,t005,Where's my package? Was supposed to arrive yes...,order_status,order_status,The customer is inquiring about the status and...,True,baseline,v1
6,t006,Tracking link in the email just spins forever....,order_status,order_status,The customer is inquiring about the tracking l...,True,baseline,v1
7,t007,"Hey, the tracking link you emailed me for orde...",order_status,order_status,The customer is inquiring about the tracking l...,True,baseline,v1
8,t008,Tracking URL is broken. Order 88421. Can you t...,order_status,order_status,The customer is inquiring about the status and...,True,baseline,v1
9,t009,Order shows delivered last Tuesday but it's no...,order_status,order_status,The customer is inquiring about the status of ...,True,baseline,v1


## 7. Evaluate: accuracy, precision, recall

- **Accuracy** = fraction of tickets classified correctly overall.
- **Precision (per class)** = of all the tickets the model *called* `refund_request`, how many actually were? High precision = few false alarms.
- **Recall (per class)** = of all the tickets that *truly were* `refund_request`, how many did the model catch? High recall = few misses.

**Why look at both:** a model that *always* predicts `other` will have 20% accuracy and 100% recall on `other` but 0% recall on everything else. Per-class precision/recall exposes that immediately.

In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def evaluate(df: pd.DataFrame, label: str):
    y_true = df["true_category"]
    y_pred = df["predicted_category"]
    print(f"=== {label} ===")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.2%}  ({df['correct'].sum()}/{len(df)} correct)\n")
    print("Per-class precision / recall / F1:")
    print(classification_report(y_true, y_pred, labels=CATEGORIES, zero_division=0))
    print("Confusion matrix (rows = true, cols = predicted):")
    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=CATEGORIES),
        index=CATEGORIES, columns=CATEGORIES,
    )
    print(cm)
    return accuracy_score(y_true, y_pred)

acc_v1 = evaluate(results_v1, "Run 1 — baseline prompt")

=== Run 1 — baseline prompt ===
Accuracy: 96.67%  (145/150 correct)

Per-class precision / recall / F1:
                precision    recall  f1-score   support

  order_status       1.00      0.96      0.98        24
refund_request       1.00      0.98      0.99        42
 product_issue       0.96      1.00      0.98        24
  account_help       1.00      0.90      0.95        30
         other       0.88      1.00      0.94        30

      accuracy                           0.97       150
     macro avg       0.97      0.97      0.97       150
  weighted avg       0.97      0.97      0.97       150

Confusion matrix (rows = true, cols = predicted):
                order_status  refund_request  product_issue  account_help  \
order_status              23               0              1             0   
refund_request             0              41              0             0   
product_issue              0               0             24             0   
account_help               0   

## 8. Export to spreadsheet for validator comments

This is the **human-in-the-loop** step. Open `results_v1.csv` in Google Sheets or Excel.

Each row has an empty `validator_comment` column. Your job:

1. **Filter `correct == FALSE`** to see only the failures.
2. For each failure, write a short note in `validator_comment` — e.g. *"complaint about delivery, model called it product_issue"*, *"ambiguous, I'd accept either"*, *"label is wrong, this really is `other`"*.
3. Also scan a sample of `correct == TRUE` rows — sometimes the model gets the right label for the *wrong reason*.
4. Once you've annotated, **cluster the comments into 2–4 failure categories** in a separate tab. For example:
   - *"Confuses `order_status` with `refund_request` when the customer mentions both delivery and money"*
   - *"Calls polite thank-you messages `account_help`"*
   - *"Routes 'wrong item' as `refund_request` instead of `product_issue`"*
5. **Pick the one failure category that matters most for your use case** (the one that's most expensive if it happens in production), and bring it back to step 9.

In [9]:
results_v1_export = results_v1.copy()
results_v1_export["validator_comment"] = ""
results_v1_export["failure_category"] = ""
results_v1_export.to_csv("results_v1.csv", index=False)
print("Wrote results_v1.csv — open it in Google Sheets or Excel.")
print("Columns:", list(results_v1_export.columns))

Wrote results_v1.csv — open it in Google Sheets or Excel.
Columns: ['id', 'ticket_text', 'true_category', 'predicted_category', 'reasoning', 'correct', 'otel_run_name', 'otel_prompt_version', 'validator_comment', 'failure_category']


## 9. Tweak the prompt to fix the failure category you picked

Edit `IMPROVED_PROMPT` below to address the failure you chose. Some common moves:

- **Add disambiguation rules.** *"If the customer mentions both delivery delay AND a refund request, classify as `order_status` — the refund is downstream of the delivery problem."*
- **Add a few-shot example** of the exact failure case with the correct label.
- **Tighten a category definition.** *"`account_help` is ONLY for login/password/profile issues, not for general site bugs."*
- **Force a step.** *"First identify the customer's primary intent in one sentence, then pick the category."*

Keep the change focused on the **one failure category** you picked. If you change everything, you won't know what helped.

In [10]:
IMPROVED_PROMPT = """You are a triage system for an e-commerce support inbox.

Classify the customer's ticket into EXACTLY ONE of these categories:
- order_status: questions about where an order is, tracking, delivery ETA, or an order that is late, missing, or never arrived
- refund_request: the customer wants their money back
- product_issue: the item arrived broken, wrong, defective, or not as described
- account_help: login, password, address, payment method changes
- other: anything that doesn't fit the above (general questions, feedback, browsing)

HOW TO CLASSIFY
Classify by the ACTION the customer is asking you to take, not by the nouns that appear while they explain what happened. The words "order," "delivered," "package," "shipping," and "address" almost always appear in the customer's backstory. Backstory is context, not the request. Find the sentence where the customer states what they want done, and classify on that sentence.

Apply these in order. First match wins.

1. If the customer asks for a refund, money back, reimbursement, a credit, or to be charged back, classify as refund_request. This holds even when the ticket also describes a late delivery, a missing package, a damaged item, or an address. The delivery problem is why they want a refund; the refund is what they are asking you to do.

2. If the customer's order is late, missing, stuck in transit, or never arrived, and they have NOT asked for money back, classify as order_status. This holds even when the ticket mentions their address while explaining what happened. A customer describing a past delivery that went wrong is NOT asking you to update their account.

3. Use account_help ONLY when the customer is asking you to change or correct account information going forward: reset a password, fix a login, update the address or payment method on file for future orders. If the address is mentioned as part of explaining a delivery that already failed, it is order_status, not account_help.

4. If the item arrived broken, wrong, or not as described and no refund is requested, classify as product_issue.

EXAMPLES

Ticket: "Order #48812 was marked delivered to my address on the 3rd but I never got it. I've checked with neighbors and the building office. I'd like a refund at this point."
Answer: refund_request

Ticket: "Order #48812 was marked delivered to my address on the 3rd but nothing showed up. I've checked with my neighbors. Can you find out what happened to it?"
Answer: order_status

Ticket: "I just moved. Can you update the shipping address on my account so my next order goes to the new place?"
Answer: account_help

Return only the category key.

Ticket:
{ticket_text}
"""

agent_v2 = build_agent(IMPROVED_PROMPT)
results_v2 = run_predictions(agent_v2, tickets, run_name="improved", prompt_version="v2")

results_v2_export = results_v2.copy()
results_v2_export["validator_comment"] = ""
results_v2_export["failure_category"] = ""
results_v2_export.to_csv("results_v2.csv", index=False)

acc_v2 = evaluate(results_v2, "Run 2 — improved prompt")

Classifying improved:   0%|          | 0/150 [00:00<?, ?it/s]

/Users/arsegas2/Projects/Lesson4/.venv/lib/python3.9/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Classification(category='...status of their order.'), input_type=Classification])
  return self.__pydantic_serializer__.to_python(
Classifying improved:   1%|          | 1/150 [00:00<02:08,  1.16it/s]

Classifying improved:   1%|▏         | 2/150 [00:01<02:00,  1.23it/s]

Classifying improved:   2%|▏         | 3/150 [00:02<02:00,  1.22it/s]

/Users/arsegas2/Projects/Lesson4/.venv/lib/python3.9/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Classification(category='... that has not arrived.'), input_type=Classification])
  return self.__pydantic_serializer__.to_python(
Classifying improved:   3%|▎         | 4/150 [00:03<01:58,  1.23it/s]

...

/Users/arsegas2/Projects/Lesson4/.venv/lib/python3.9/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Classification(category='...ability of gift cards.'), input_type=Classification])
  return self.__pydantic_serializer__.to_python(
Classifying improved:  99%|█████████▊| 148/150 [02:06<00:01,  1.25it/s]

Classifying improved:  99%|█████████▉| 149/150 [02:06<00:00,  1.28it/s]

Classifying improved: 100%|██████████| 150/150 [02:08<00:00,  1.01s/it]

Classifying improved: 100%|██████████| 150/150 [02:08<00:00,  1.17it/s]

=== Run 2 — improved prompt ===
Accuracy: 96.00%  (144/150 correct)

Per-class precision / recall / F1:
                precision    recall  f1-score   support

  order_status       1.00      0.96      0.98        24
refund_request       1.00      0.98      0.99        42
 product_issue       1.00      1.00      1.00        24
  account_help       1.00      0.87      0.93        30
         other       0.83      1.00      0.91        30

      accuracy                           0.96       150
     macro avg       0.97      0.96      0.96       150
  weighted avg       0.97      0.96      0.96       150

Confusion matrix (rows = true, cols = predicted):
                order_status  refund_request  product_issue  account_help  \
order_status              23               0              0             0   
refund_request             0              41              0             0   
product_issue              0               0             24             0   
account_help               0   

## 10. Compare the two runs

Now look at the headline accuracy and at *which specific tickets flipped* between runs. Some will go from wrong → right (the win you were aiming for). Some may go from right → wrong (a regression you caused). This is normal — almost no prompt change is strictly Pareto-better, and the trade-offs are the most important thing to understand.

In [11]:
print(f"Accuracy v1: {acc_v1:.2%}")
print(f"Accuracy v2: {acc_v2:.2%}")
print(f"Δ          : {(acc_v2 - acc_v1):+.2%}\n")

comparison = results_v1.merge(
    results_v2[["id", "predicted_category", "reasoning", "correct"]],
    on="id", suffixes=("_v1", "_v2"),
)

flipped = comparison[comparison["predicted_category_v1"] != comparison["predicted_category_v2"]]
print(f"{len(flipped)} tickets changed prediction between runs.\n")

wins   = flipped[(~flipped["correct_v1"]) & (flipped["correct_v2"])]
losses = flipped[(flipped["correct_v1"]) & (~flipped["correct_v2"])]
print(f"Wins (wrong → right):     {len(wins)}")
print(f"Regressions (right → wrong): {len(losses)}")

flipped[["ticket_text", "true_category", "predicted_category_v1", "predicted_category_v2"]]

Accuracy v1: 96.67%
Accuracy v2: 96.00%
Δ          : -0.67%

3 tickets changed prediction between runs.

Wins (wrong → right):     1
Regressions (right → wrong): 2


,ticket_text,true_category,predicted_category_v1,predicted_category_v2
6,Tracking link in the email just spins forever....,order_status,order_status,other
25,Carrier delivered to the wrong address. The ad...,order_status,product_issue,order_status
100,2FA codes are showing up that I didn't ask for...,account_help,account_help,other


## 11. What to take away

- **LangGraph** gave us a clean shape for the agent. Right now it's one node, but you can drop in retrieval, tools, or a self-check node without rewriting the eval harness.
- **LangSmith** turned every LLM call into an inspectable trace. OpenTelemetry added a standard parent span around each ticket so CSV rows, prompt versions, labels, and correctness are attached to the trace.
- **Accuracy alone is a trap.** Per-class precision/recall and the confusion matrix tell you *what kind* of mistakes the model is making.
- **Validator comments are the most valuable artifact in this whole notebook.** Numbers tell you *that* something is wrong; human notes tell you *what* and *why*.
- **Prompt iteration is a measure → diagnose → fix → re-measure loop.** Without the dataset and the metrics, prompt tweaks are just vibes.

### Suggested next steps
- Replace the synthetic seed tickets with anonymized **real** tickets from your inbox.
- Upload the dataset to LangSmith as a versioned **Dataset** and use the LangSmith `evaluate()` runner so each prompt version gets a saved score.
- Add a second node to the graph — e.g., a confidence check that routes low-confidence predictions to a human queue.